# Part 1: Data Preparation and Exploratory Analysis

## The Kulturkampf and Catholic Fertility in Prussia

**Research question:** Did Bismarck's anti-Catholic Kulturkampf legislation (1872–1878) affect the Catholic–Protestant fertility differential in Prussian counties?

This first notebook handles loading the raw Galloway Prussia Database (1861–1914), harmonizing variables across cross-sections, interpolating missing population data, and generating the core panel dataset used throughout the rest of the analysis.


### 1. Environment Setup
We load the necessary paths, external libraries, and our custom `src.data` pipeline functions.


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Paths
DATA_RAW = project_root / "data" / "raw" / "galloway_data"
DATA_PROCESSED = project_root / "data" / "processed"
OUTPUTS = project_root / "outputs" / "figures"
OUTPUTS.mkdir(exist_ok=True, parents=True)

print("Setup complete. Outputs will be saved to:", OUTPUTS)

from src.data.load_data import load_rel1871, load_vit_panel, load_ipehd_master
from src.data.build_dataset import build_analysis_panel
from src.visualization.plots import plot_fertility_trends, plot_cath_distribution


### 2. Building the Analysis Panel
We join the time-invariant REL1871 religious census data (providing the baseline Catholic share for each county) with the annual VIT vital registration panel. We then compute crude birth rates, marriage rates, and other demographic outcomes.


In [ ]:
panel = build_analysis_panel(
    data_dir=DATA_RAW,
    year_start=1862,
    year_end=1890,
    save=True,  # Saves to data/processed/analysis_panel.parquet
)
panel.head(10)



### 3. Descriptive Statistics
Let's examine the raw means and standard deviations of our primary outcomes, split by pre- vs. post-Kulturkampf periods and by religious composition (High >50% Catholic vs. Low Catholic). This provides a foundational intuition before moving to rigorous econometric models.


In [ ]:
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

for period, label in [(panel["Year"] < 1873, "Pre-Kulturkampf (1862-1872)"),
                       (panel["Year"] >= 1873, "Post-Kulturkampf (1873-1890)")]:
    sub = panel[period]
    print(f"\n{label}:")
    print(f"  N = {len(sub)}, Counties = {sub['Code'].nunique()}")
    for var in ["cbr", "legitimate_br", "marriage_rate"]:
        if var in sub.columns and sub[var].notna().any():
            print(f"  {var}: mean={sub[var].mean():.2f}, sd={sub[var].std():.2f}")

print(f"\n{'='*60}")
print("BY RELIGIOUS COMPOSITION (full sample)")
print(f"{'='*60}")
grouped = panel.groupby("high_cath").agg(
    n_counties=("Code", "nunique"),
    mean_cath_share=("cath_share", "mean"),
    mean_cbr=("cbr", "mean"),
    mean_marriage_rate=("marriage_rate", lambda x: x.mean() if x.notna().any() else np.nan),
).round(2)
grouped.index = ["Low Catholic (≤50%)", "High Catholic (>50%)"]
print(grouped)



### 4. Visualizing Demographics and the Treatment Variable
The Kulturkampf was a nationwide policy, but its localized intensity depended on the Catholic share of a county. We map out the distribution of Catholic shares to ensure there is sufficient variation for our continuous and binary treatment definitions.


In [ ]:
fig, ax = plot_cath_distribution(
    panel,
    savepath=str(OUTPUTS / "fig1_cath_distribution.png"),
)
plt.show()



### 5. Fertility and Marriage Trends Over Time
Before running fixed-effects models, plotting raw trends is vital. Do high-Catholic counties have permanently higher fertility? Do their trajectories diverge after 1872?


In [ ]:
fig, ax = plot_fertility_trends(
    panel,
    outcome="cbr",
    ylabel="Crude birth rate (per 1,000)",
    title="Fertility trends: High- vs Low-Catholic counties",
    savepath=str(OUTPUTS / "fig2_fertility_trends.png"),
)
plt.show()

# Focus on legitimate births (only reliably available post-1875)
panel_post75 = panel[panel["Year"] >= 1875].copy()
fig, ax = plot_fertility_trends(
    panel_post75,
    outcome="legitimate_br",
    ylabel="Legitimate birth rate (per 1,000)",
    title="Legitimate fertility trends: High- vs Low-Catholic counties",
    savepath=str(OUTPUTS / "fig3_legit_fertility_trends.png"),
)
plt.show()

# Catholic marriage share
fig, ax = plot_fertility_trends(
    panel_post75,
    outcome="cath_marriage_share",
    ylabel="Catholic marriages (% of total)",
    title="Catholic marriage share over time",
    savepath=str(OUTPUTS / "fig4_cath_marriages.png"),
)
plt.show()

